# 03 — Pipeline RAG complet

**Objectif** : combiner retrieval + LLM Claude pour répondre en langage naturel avec citations.

Ce notebook couvre le point **2.3** (suite) du formulaire SpeciTec.

**Prérequis** : créer un fichier `.env` à la racine avec :
```
ANTHROPIC_API_KEY=sk-ant-...
```

In [ ]:
import sys, warnings, logging
from pathlib import Path
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
logging.disable(logging.WARNING)

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / '.env')

from src.retriever import JobRetriever
from src.rag import JobRAG

retriever = JobRetriever()
rag = JobRAG(retriever)

print('Pipeline RAG initialisé')
print(f'Index : {retriever.index.ntotal} documents')
print(f'LLM   : {rag.llm_model}')

## Question 1 — Requête techno + lieu

In [ ]:
result = rag.ask("Quelles offres demandent dbt et sont basées à Genève ?", verbose=True)

## Question 2 — Requête négative (sans cloud)

In [ ]:
rag.ask_pretty("Y a-t-il des postes de Data Engineer qui ne demandent pas de cloud ?")

## Question 3 — Comparaison BI (Power BI vs Tableau)

In [ ]:
rag.ask_pretty("Compare les offres BI qui demandent Power BI vs Tableau")

## Question 4 — Recruteurs bancaires

In [ ]:
rag.ask_pretty("Quelles entreprises recrutent des profils Oracle en Suisse romande ?")

## Question 5 — Offres récentes

In [ ]:
rag.ask_pretty("Quelles sont les offres les plus récentes en data engineering ?")

## Cas limite — Aucun résultat pertinent

In [ ]:
# Question hors du domaine — doit retourner une réponse honnête
rag.ask_pretty("Quelles offres de chef cuisinier sont disponibles à Lausanne ?")

## Recherche avec filtre par catégorie

In [ ]:
# On filtre explicitement sur BI_ANALYTICS avant d'envoyer au LLM
rag.ask_pretty(
    "Quelles sont les compétences les plus demandées pour les postes BI ?",
    filters={"label": "BI_ANALYTICS"}
)

---

## Architecture du pipeline

```
Question utilisateur
        ↓
Embedding (sentence-transformers, 384 dims)
        ↓
FAISS IndexFlatIP — top-20 candidats
        ↓
Re-ranking (sim × recency × category_weight) → top-5
        ↓
Prompt template (system + offres + question)
        ↓
Claude API (claude-sonnet-4-6)
        ↓
Réponse avec citations [Titre - Entreprise]
```

**Prochaine étape** : `04_evaluation.ipynb` — mesure de la qualité